In [ ]:
pip install pandas yfinance pandas_ta


In [ ]:
import pandas as pd
import yfinance as yf
import numpy as np

symbol = "TITAN.NS"
df = yf.download(symbol, start="2024-01-01")

# True Range
df['TR'] = np.maximum(
    df['High'] - df['Low'],
    np.maximum(
        abs(df['High'] - df['Close'].shift(1)),
        abs(df['Low'] - df['Close'].shift(1))
    )
)

# Directional Movement
df['+DM'] = np.where(
    (df['High'] - df['High'].shift(1)) > (df['Low'].shift(1) - df['Low']),
    np.maximum(df['High'] - df['High'].shift(1), 0),
    0
)

df['-DM'] = np.where(
    (df['Low'].shift(1) - df['Low']) > (df['High'] - df['High'].shift(1)),
    np.maximum(df['Low'].shift(1) - df['Low'], 0),
    0
)

# Wilder smoothing
period = 14
df['TR_smooth'] = df['TR'].rolling(period).mean()
df['+DM_smooth'] = df['+DM'].rolling(period).mean()
df['-DM_smooth'] = df['-DM'].rolling(period).mean()

# DI values
df['+DI'] = 100 * (df['+DM_smooth'] / df['TR_smooth'])
df['-DI'] = 100 * (df['-DM_smooth'] / df['TR_smooth'])

# DX & ADX
df['DX'] = 100 * abs(df['+DI'] - df['-DI']) / (df['+DI'] + df['-DI'])
df['ADX'] = df['DX'].rolling(period).mean()

print(df[['Close', '+DI', '-DI', 'ADX']].tail())



In [ ]:
df = df.dropna().reset_index(drop=True)

adx = df['ADX'].iloc[-1]
plus_di = df['+DI'].iloc[-1]
minus_di = df['-DI'].iloc[-1]


In [ ]:
adx = float(adx)
plus_di = float(plus_di)
minus_di = float(minus_di)


In [ ]:
if adx > 25:
    if plus_di > minus_di:
        print("📈 Strong Uptrend → Buy / Hold")
    else:
        print("📉 Strong Downtrend → Sell / Short")
else:
    print("⚠️ Sideways Market → Avoid")
